<a href="https://colab.research.google.com/github/elvissoares/EQE595-SimMol/blob/main/notebooks/8_Proteina.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Para usar o OpenMM no Google Colab devemos fazer o seguinte passo:
1. Instalar o `openmm[cuda12]`, `mdtraj` e `pdbfixer` via `pip`

In [ ]:
!pip install openmm[cuda12] mdtraj pdbfixer

2. Testando se a instalação deu certo e quais `Platform` estão disponíveis

In [ ]:
!python -m openmm.testInstallation

# Aula Prática 08 - Proteína em Água

Autor: [Prof. Elvis do A. Soares](https://github.com/elvissoares)

Contato: [elvis@peq.coppe.ufrj.br](mailto:elvis@peq.coppe.ufrj.br) - [Programa de Engenharia Química, PEQ/COPPE, UFRJ, Brasil](https://www.peq.coppe.ufrj.br/)

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from openmm.app import *
from openmm import *
from openmm.unit import *
from sys import stdout

# 0. Abrindo Google Drive para salvar arquivos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# path to output files
path = '/content/drive/MyDrive/SimMol/'

# 1. Parâmetros principais

In [ ]:
pdb_id = '1AKI'

temperature = 300.0 * kelvin
pressure = 1.0 * atmosphere
ph = 7.0

ionic_strength = 0.15 * molar

padding = 1.0 * nanometer

timestep = 2.0 * femtoseconds
friction = 1.0 / picosecond

minimization_tolerance = 10.0 * kilojoule_per_mole / nanometer
max_minimization_iterations = 5000

nvt_steps = 50_000       # 100 ps com dt = 2 fs
npt_steps = 100_000      # 200 ps
production_steps = 500_000  # 1 ns

report_interval = 1000
trajectory_interval = 1000

output_dir = path + f"md_{pdb_id}"
os.makedirs(output_dir, exist_ok=True)

# 2. Baixar e corrigir PDB

In [ ]:
print(f"Preparando PDB: {pdb_id}")

!wget https://files.rcsb.org/download/{pdb_id}.pdb

from pdbfixer import PDBFixer

fixer = PDBFixer(pdbid=pdb_id)

# Remove heteroátomos não essenciais.
# keepWater=False remove águas cristalográficas.
fixer.removeHeterogens(keepWater=False)

# Identifica e adiciona resíduos/átomos faltantes quando possível.
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()

# Adiciona hidrogênios de acordo com pH.
fixer.addMissingHydrogens(pH=ph)

fixed_pdb_path = os.path.join(output_dir, f"{pdb_id}_fixed.pdb")

with open(fixed_pdb_path, "w") as f:
    PDBFile.writeFile(fixer.topology, fixer.positions, f)

print(f"PDB corrigido salvo em: {fixed_pdb_path}")

# 3. Campo de forças: Amber14 + TIP4P-Ew

In [ ]:
forcefield = ForceField(
    "amber14-all.xml",
    "amber14/tip4pew.xml",
)

# 4. Solvatação com TIP4P-Ew

In [ ]:
modeller = Modeller(fixer.topology, fixer.positions)

modeller.addSolvent(
    forcefield,
    model="tip4pew",
    padding=padding,
    ionicStrength=ionic_strength,
    neutralize=True,
)

# Necessário para modelos de água com sítio virtual, como TIP4P-Ew.
modeller.addExtraParticles(forcefield)

solvated_pdb_path = os.path.join(output_dir, f"{pdb_id}_solvated_tip4pew.pdb")

with open(solvated_pdb_path, "w") as f:
    PDBFile.writeFile(modeller.topology, modeller.positions, f)

print(f"Sistema solvatado salvo em: {solvated_pdb_path}")

# 5. Criar sistema OpenMM

In [ ]:
system = forcefield.createSystem(
    modeller.topology,
    nonbondedMethod=PME,
    nonbondedCutoff=1.0 * nanometer,
    constraints=HBonds,
    rigidWater=True,
    ewaldErrorTolerance=1.0e-4,
)

# Barostato para etapa NPT e produção NPT.
barostat = MonteCarloBarostat(pressure, temperature, 25)
system.addForce(barostat)

# 6. Integrador e plataforma

In [ ]:
integrator = LangevinMiddleIntegrator(
    temperature,
    friction,
    timestep,
)

# Tente CUDA; se não houver GPU compatível, use CPU.
try:
    platform = Platform.getPlatformByName("CUDA")
    platform_properties = {
        "Precision": "mixed",
        "DeviceIndex": "0",
    }
    print("Usando plataforma CUDA.")
except Exception:
    platform = Platform.getPlatformByName("CPU")
    platform_properties = {}
    print("CUDA não disponível. Usando CPU.")


simulation = Simulation(
    modeller.topology,
    system,
    integrator,
    platform,
    platform_properties,
)

simulation.context.setPositions(modeller.positions)

# 7. Minimização

In [ ]:
print("Minimizando energia...")

simulation.minimizeEnergy(
    tolerance=minimization_tolerance,
    maxIterations=max_minimization_iterations,
)

minimized_pdb_path = os.path.join(output_dir, f"{pdb_id}_minimized.pdb")

state = simulation.context.getState(getPositions=True, getEnergy=True)
with open(minimized_pdb_path, "w") as f:
    PDBFile.writeFile(
        simulation.topology,
        state.getPositions(),
        f,
    )

print(f"Energia potencial após minimização: {state.getPotentialEnergy()}")
print(f"Estrutura minimizada salva em: {minimized_pdb_path}")

# 8. Equilibração NVT

In [ ]:
print("Equilibrando em NVT...")

# Durante NVT, desligamos o barostato temporariamente colocando frequência 0.
barostat.setFrequency(0)

simulation.context.setVelocitiesToTemperature(temperature)

simulation.reporters.append(
    StateDataReporter(
        os.path.join(output_dir, "nvt.log"),
        report_interval,
        step=True,
        time=True,
        potentialEnergy=True,
        kineticEnergy=True,
        totalEnergy=True,
        temperature=True,
        speed=True,
        separator=",",
    )
)

simulation.step(nvt_steps)

nvt_state_path = os.path.join(output_dir, f"{pdb_id}_after_nvt.xml")
simulation.saveState(nvt_state_path)

print(f"Estado após NVT salvo em: {nvt_state_path}")

# 9. Equilibração NPT

In [ ]:
print("Equilibrando em NPT...")

barostat.setFrequency(25)

simulation.reporters.clear()

simulation.reporters.append(
    StateDataReporter(
        os.path.join(output_dir, "npt.log"),
        report_interval,
        step=True,
        time=True,
        potentialEnergy=True,
        kineticEnergy=True,
        totalEnergy=True,
        temperature=True,
        volume=True,
        density=True,
        speed=True,
        separator=",",
    )
)

simulation.step(npt_steps)

npt_state_path = os.path.join(output_dir, f"{pdb_id}_after_npt.xml")
simulation.saveState(npt_state_path)

state = simulation.context.getState(getPositions=True)
npt_pdb_path = os.path.join(output_dir, f"{pdb_id}_after_npt.pdb")

with open(npt_pdb_path, "w") as f:
    PDBFile.writeFile(simulation.topology, state.getPositions(), f)

print(f"Estado após NPT salvo em: {npt_state_path}")
print(f"Configuração após NPT salva em: {npt_pdb_path}")

# 10. Produção

In [ ]:
print("Rodando produção NPT...")

simulation.reporters.clear()

dcd_path = os.path.join(output_dir, f"{pdb_id}_production.dcd")
prod_log_path = os.path.join(output_dir, "production.log")
checkpoint_path = os.path.join(output_dir, f"{pdb_id}.chk")

simulation.reporters.append(
    DCDReporter(
        dcd_path,
        trajectory_interval,
    )
)

simulation.reporters.append(
    StateDataReporter(
        prod_log_path,
        report_interval,
        step=True,
        time=True,
        potentialEnergy=True,
        kineticEnergy=True,
        totalEnergy=True,
        temperature=True,
        volume=True,
        density=True,
        speed=True,
        remainingTime=True,
        totalSteps=production_steps,
        separator=",",
    )
)

simulation.reporters.append(
    CheckpointReporter(
        checkpoint_path,
        10_000,
    )
)

simulation.step(production_steps)

final_state_path = os.path.join(output_dir, f"{pdb_id}_final_state.xml")
final_pdb_path = os.path.join(output_dir, f"{pdb_id}_final.pdb")

simulation.saveState(final_state_path)

state = simulation.context.getState(getPositions=True, getEnergy=True)
with open(final_pdb_path, "w") as f:
    PDBFile.writeFile(simulation.topology, state.getPositions(), f)

print("Simulação finalizada.")
print(f"Trajetória: {dcd_path}")
print(f"Log de produção: {prod_log_path}")
print(f"Checkpoint: {checkpoint_path}")
print(f"Estado final: {final_state_path}")
print(f"PDB final: {final_pdb_path}")

# 11. Analisando a estatística dos dados

In [ ]:
df = pd.read_csv(f"{output_dir}/production.log")

In [ ]:
df.head()

In [ ]:
fig, axs = plt.subplots(3, 1, sharex=True)

axs[0].plot(df['Time (ps)'],df['Potential Energy (kJ/mole)'],color='C0',label='U')
axs[0].legend(loc='best')
axs[0].set_ylabel('U (kJ/mole)')

axs[1].plot(df['Time (ps)'],df['Temperature (K)'],color='C3',label='T')
axs[1].legend(loc='best')
axs[1].set_ylabel('T (K)')

axs[2].plot(df['Time (ps)'],df['Box Volume (nm^3)'],color='C1',label='V')
axs[2].legend(loc='best')
axs[2].set_xlabel('Time (ps)')
axs[2].set_ylabel('Volume (nm$^3$)')

# 12. MDTraj

In [ ]:
import mdtraj as md

traj = md.load(
    f"md_{pdb_id}/{pdb_id}_production.dcd",
    top=f"md_{pdb_id}/{pdb_id}_after_npt.pdb",
)

protein_atoms = traj.topology.select("protein")
ca_atoms = traj.topology.select("name CA")
backbone_atoms = traj.topology.select("backbone")

traj.superpose(traj, 0, atom_indices=backbone_atoms)

rmsd_ca = md.rmsd(traj, traj, 0, atom_indices=ca_atoms)
rmsd_backbone = md.rmsd(traj, traj, 0, atom_indices=backbone_atoms)
rmsd_protein = md.rmsd(traj, traj, 0, atom_indices=protein_atoms)

time_ns = traj.time / 1000.0

plt.plot(time_ns, rmsd_ca, label="C-alpha")
plt.plot(time_ns, rmsd_backbone, label="Backbone")
plt.plot(time_ns, rmsd_protein, label="Protein")
plt.xlabel("Tempo / ns")
plt.ylabel("RMSD / nm")
plt.legend()
plt.tight_layout()
plt.show()

print("RMSD C-alpha médio / nm:", np.mean(rmsd_ca))
print("RMSD backbone médio / nm:", np.mean(rmsd_backbone))
print("RMSD proteína médio / nm:", np.mean(rmsd_protein))

> <span style="color:#A03;font-size:14pt">
> &#x270B; HANDS-ON! &#x1F528;
> </span>
> 
> - O RMSD atinge um platô?
> - Qual é o valor médio do RMSD após a equilibração?
> - Esse valor é compatível com estabilidade estrutural?

Quais regiões da proteína são mais flexíveis?

In [ ]:
traj.superpose(traj, 0, atom_indices=backbone_atoms)

rmsf_ca = md.rmsf(traj, traj, frame=0, atom_indices=ca_atoms)

ca_residues = [traj.topology.atom(i).residue for i in ca_atoms]
residue_ids = [res.index for res in ca_residues]
residue_names = [str(res) for res in ca_residues]

plt.plot(residue_ids, rmsf_ca, marker="o")
plt.xlabel("Índice do resíduo")
plt.ylabel("RMSF C-alpha / nm")
plt.tight_layout()
plt.show()

top3 = np.argsort(rmsf_ca)[-3:][::-1]

print("Resíduos mais flexíveis:")
for idx in top3:
    print(residue_names[idx], rmsf_ca[idx], "nm")

A proteína permanece compacta ou expande durante a simulação?

In [ ]:
protein_traj = traj.atom_slice(protein_atoms)

rg = md.compute_rg(protein_traj)

plt.plot(time_ns, rg)
plt.xlabel("Tempo / ns")
plt.ylabel("Raio de giro / nm")
plt.tight_layout()
plt.show()

print("Rg médio / nm:", np.mean(rg))
print("Rg desvio padrão / nm:", np.std(rg))